<a href="https://colab.research.google.com/github/alxmzr/Colab/blob/main/Heikin_ashi_MACD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import yfinance as yf
import pandas as pd
import numpy as np
import itertools

def backtest_smoothed_macd(data, smooth_period, fast, slow, signal, commission=0.001):
    df = data.copy()

    # 1. Сглаживание цен для Smoothed Heikin Ashi
    df['sOpen'] = df['Open'].ewm(span=smooth_period, adjust=False).mean()
    df['sHigh'] = df['High'].ewm(span=smooth_period, adjust=False).mean()
    df['sLow'] = df['Low'].ewm(span=smooth_period, adjust=False).mean()
    df['sClose'] = df['Close'].ewm(span=smooth_period, adjust=False).mean()

    # 2. Расчет Heikin Ashi на сглаженных данных
    df_ha = pd.DataFrame(index=df.index)
    df_ha['HA_Close'] = (df['sOpen'] + df['sHigh'] + df['sLow'] + df['sClose']) / 4

    ha_open = np.zeros(len(df))
    ha_open[0] = (df['sOpen'].iloc[0] + df['sClose'].iloc[0]) / 2
    for i in range(1, len(df)):
        ha_open[i] = (ha_open[i-1] + df_ha['HA_Close'].iloc[i-1]) / 2
    df_ha['HA_Open'] = ha_open

    # 3. Расчет MACD
    df['EMA_fast'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df['EMA_slow'] = df['Close'].ewm(span=slow, adjust=False).mean()
    df['MACD'] = df['EMA_fast'] - df['EMA_slow']
    df['Signal_Line'] = df['MACD'].ewm(span=signal, adjust=False).mean()

    # Условие: MACD выше сигнальной И свеча Smoothed HA бычья
    condition = (df['MACD'] > df['Signal_Line']) & (df_ha['HA_Close'] > df_ha['HA_Open'])
    df['Position'] = np.where(condition, 1, 0)

    # Расчет доходности с учетом комиссий
    df['Market_Returns'] = df['Close'].pct_change()
    df['Strategy_Returns'] = df['Position'].shift(1) * df['Market_Returns']

    # Симуляция транзакционных издержек (комиссия при каждой смене позиции)
    df['Trades'] = df['Position'].diff().abs()
    df['Strategy_Returns'] -= df['Trades'] * commission

    return (1 + df['Strategy_Returns'].fillna(0)).prod()

def optimize_smoothed_strategy(ticker, period='2y'):
    print(f"\nОптимизация Smoothed HA + MACD для: {ticker} (с учетом комиссии 0.1%)")
    data = yf.download(ticker, period=period, interval='1d', progress=False)
    if data.empty: return None
    if isinstance(data.columns, pd.MultiIndex): data.columns = data.columns.get_level_values(0)

    # Сетки параметров
    smooth_ranges = [5]
    fast_ranges = [15]
    slow_ranges = [20]
    signal_ranges = [9, 12]

    best_ret = 0
    best_p = None

    for sm, f, s, sig in itertools.product(smooth_ranges, fast_ranges, slow_ranges, signal_ranges):
        if f >= s: continue
        res = backtest_smoothed_macd(data, sm, f, s, sig, commission=0.001)
        if res > best_ret:
            best_ret = res
            best_p = (sm, f, s, sig)

    print(f"Лучшие (Smooth, Fast, Slow, Sig): {best_p}")
    print(f"Доходность с комиссией: {(best_ret-1)*100:.2f}%")
    return best_p

assets = ['BTC-USD', 'ETH-USD', 'EURUSD=X', 'GBPUSD=X']
for asset in assets:
    optimize_smoothed_strategy(asset)


Оптимизация Smoothed HA + MACD для: BTC-USD (с учетом комиссии 0.1%)
Лучшие (Smooth, Fast, Slow, Sig): (5, 15, 20, 12)
Доходность с комиссией: 9.53%

Оптимизация Smoothed HA + MACD для: ETH-USD (с учетом комиссии 0.1%)


/tmp/ipykernel_2823/1761466018.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)
/tmp/ipykernel_2823/1761466018.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучшие (Smooth, Fast, Slow, Sig): (5, 15, 20, 12)
Доходность с комиссией: -7.48%

Оптимизация Smoothed HA + MACD для: EURUSD=X (с учетом комиссии 0.1%)
Лучшие (Smooth, Fast, Slow, Sig): (5, 15, 20, 12)
Доходность с комиссией: -4.16%

Оптимизация Smoothed HA + MACD для: GBPUSD=X (с учетом комиссии 0.1%)


/tmp/ipykernel_2823/1761466018.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)
/tmp/ipykernel_2823/1761466018.py:47: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучшие (Smooth, Fast, Slow, Sig): (5, 15, 20, 9)
Доходность с комиссией: 1.26%


In [12]:
def backtest_macd(data, fast, slow, signal, commission=0.001):
    df = data.copy()

    # Calculate MACD
    df['EMA_fast'] = df['Close'].ewm(span=fast, adjust=False).mean()
    df['EMA_slow'] = df['Close'].ewm(span=slow, adjust=False).mean()
    df['MACD'] = df['EMA_fast'] - df['EMA_slow']
    df['Signal_Line'] = df['MACD'].ewm(span=signal, adjust=False).mean()

    # Generate trading signals (1 for buy, 0 for hold/sell)
    # Buy when MACD crosses above Signal Line
    df['Position'] = np.where(df['MACD'] > df['Signal_Line'], 1, 0)

    # Calculate market and strategy returns
    df['Market_Returns'] = df['Close'].pct_change()
    df['Strategy_Returns'] = df['Position'].shift(1) * df['Market_Returns']

    # Simulate transaction costs (commission on every position change)
    df['Trades'] = df['Position'].diff().abs()
    df['Strategy_Returns'] -= df['Trades'] * commission

    return (1 + df['Strategy_Returns'].fillna(0)).prod()

In [13]:
def backtest_only_ha(data, smooth_period, commission=0.001):
    df = data.copy()

    # 1. Сглаживание цен
    df['sOpen'] = df['Open'].ewm(span=smooth_period, adjust=False).mean()
    df['sHigh'] = df['High'].ewm(span=smooth_period, adjust=False).mean()
    df['sLow'] = df['Low'].ewm(span=smooth_period, adjust=False).mean()
    df['sClose'] = df['Close'].ewm(span=smooth_period, adjust=False).mean()

    # 2. Расчет Heikin Ashi
    ha_close = (df['sOpen'] + df['sHigh'] + df['sLow'] + df['sClose']) / 4
    ha_open = np.zeros(len(df))
    ha_open[0] = (df['sOpen'].iloc[0] + df['sClose'].iloc[0]) / 2
    for i in range(1, len(df)):
        ha_open[i] = (ha_open[i-1] + ha_close.iloc[i-1]) / 2

    # Сигнал: Только цвет свечи
    df['Position'] = np.where(ha_close > ha_open, 1, 0)

    # Доходность
    df['Market_Returns'] = df['Close'].pct_change()
    df['Strategy_Returns'] = df['Position'].shift(1) * df['Market_Returns']
    df['Trades'] = df['Position'].diff().abs()
    df['Strategy_Returns'] -= df['Trades'] * commission

    return (1 + df['Strategy_Returns'].fillna(0)).prod()

def optimize_pure_ha(ticker, period='2y'):
    print(f"\nОптимизация Pure Smoothed HA для: {ticker}")
    data = yf.download(ticker, period=period, interval='1d', progress=False)
    if data.empty: return
    if isinstance(data.columns, pd.MultiIndex): data.columns = data.columns.get_level_values(0)

    smooth_ranges = range(2, 21) # Проверяем период сглаживания от 2 до 20
    best_ret = 0
    best_smooth = None

    for sm in smooth_ranges:
        res = backtest_only_ha(data, sm)
        if res > best_ret:
            best_ret = res
            best_smooth = sm

    print(f"Лучший период сглаживания: {best_smooth}")
    print(f"Итоговая доходность: {(best_ret-1)*100:.2f}%")

for asset in assets:
    optimize_pure_ha(asset)


Оптимизация Pure Smoothed HA для: BTC-USD


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 14
Итоговая доходность: 60.61%

Оптимизация Pure Smoothed HA для: ETH-USD


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 14
Итоговая доходность: 8.22%

Оптимизация Pure Smoothed HA для: EURUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 19
Итоговая доходность: 6.58%

Оптимизация Pure Smoothed HA для: GBPUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 8
Итоговая доходность: 5.39%


In [14]:
# Оптимизация только Heikin Ashi для Forex активов
forex_assets = ['EURUSD=X', 'GBPUSD=X']

print("--- Оптимизация Pure Smoothed HA для Форекс ---")
for asset in forex_assets:
    optimize_pure_ha(asset, period='2y')


--- Оптимизация Pure Smoothed HA для Форекс ---

Оптимизация Pure Smoothed HA для: EURUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 19
Итоговая доходность: 6.58%

Оптимизация Pure Smoothed HA для: GBPUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 8
Итоговая доходность: 5.39%


In [15]:
import pandas as pd

ticker = 'EURUSD=X'
data = yf.download(ticker, period='2y', interval='1d', progress=False)
if isinstance(data.columns, pd.MultiIndex): data.columns = data.columns.get_level_values(0)

# Best params for HA+MACD found previously: (5, 15, 20, 9)
ret_ha_macd = backtest_smoothed_macd(data, 5, 15, 20, 9, commission=0.001)

# Best params for Pure HA found previously: 19
ret_pure_ha = backtest_only_ha(data, 19, commission=0.001)

print(f"--- Comparison for {ticker} (2 Years, Daily) ---")
print(f"HA + MACD (5, 15, 20, 9) Return: {(ret_ha_macd-1)*100:.2f}%")
print(f"Pure Smoothed HA (19) Return:   {(ret_pure_ha-1)*100:.2f}%")

--- Comparison for EURUSD=X (2 Years, Daily) ---
HA + MACD (5, 15, 20, 9) Return: -4.56%
Pure Smoothed HA (19) Return:   6.58%


/tmp/ipykernel_2823/3130367357.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='2y', interval='1d', progress=False)


In [16]:
specific_params = (5, 10, 15, 9)  # Smooth, Fast, Slow, Signal
forex_assets = ['EURUSD=X', 'GBPUSD=X']

print(f"Проверка настроек {specific_params} на таймфрейме 1 ЧАС:\n")

for asset in forex_assets:
    # Для интервала 1h yfinance позволяет загрузить до 730 дней
    data = yf.download(asset, period='730d', interval='1h', progress=False)
    if not data.empty:
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        res = backtest_smoothed_macd(data, *specific_params)
        print(f"{asset}: Итоговая доходность = {(res-1)*100:.2f}%")

Проверка настроек (5, 10, 15, 9) на таймфрейме 1 ЧАС:



/tmp/ipykernel_2823/3503243019.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='730d', interval='1h', progress=False)


EURUSD=X: Итоговая доходность = -86.11%


/tmp/ipykernel_2823/3503243019.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='730d', interval='1h', progress=False)


GBPUSD=X: Итоговая доходность = -86.21%


In [17]:
def optimize_hourly_strategy(ticker):
    print(f"\n--- Оптимизация 1H для: {ticker} ---")
    # Загружаем 730 дней часовых данных
    data = yf.download(ticker, period='730d', interval='1h', progress=False)
    if data.empty: return
    if isinstance(data.columns, pd.MultiIndex): data.columns = data.columns.get_level_values(0)

    # Расширенная сетка для поиска на часе
    smooth_ranges = [3, 5, 8]
    fast_ranges = [8, 10, 12]
    slow_ranges = [20, 24, 28]
    signal_ranges = [7, 9, 12]

    best_ret = 0
    best_p = None

    for sm, f, s, sig in itertools.product(smooth_ranges, fast_ranges, slow_ranges, signal_ranges):
        if f >= s: continue
        res = backtest_smoothed_macd(data, sm, f, s, sig)
        if res > best_ret:
            best_ret = res
            best_p = (sm, f, s, sig)

    print(f"Лучшие параметры (Smooth, Fast, Slow, Sig): {best_p}")
    print(f"Доходность: {(best_ret-1)*100:.2f}%")

for asset in forex_assets:
    optimize_hourly_strategy(asset)


--- Оптимизация 1H для: EURUSD=X ---


/tmp/ipykernel_2823/2981015827.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Лучшие параметры (Smooth, Fast, Slow, Sig): (8, 12, 28, 12)
Доходность: -77.19%

--- Оптимизация 1H для: GBPUSD=X ---


/tmp/ipykernel_2823/2981015827.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Лучшие параметры (Smooth, Fast, Slow, Sig): (8, 12, 28, 12)
Доходность: -80.20%


In [18]:
forex_list = ['EURUSD=X', 'GBPUSD=X', 'AUDUSD=X', 'NZDUSD=X', 'USDCHF=X', 'USDCAD=X', 'USDJPY=X']
indices_list = ['^IXIC', '^GSPC', '^DJI'] # NASDAQ, S&P 500, Dow Jones

all_requested_assets = forex_list + indices_list

print("--- Optimization for Forex and Indices (Pure Heikin Ashi) ---")
for ticker in all_requested_assets:
    try:
        optimize_pure_ha(ticker, period='2y')
    except Exception as e:
        print(f"Could not optimize for {ticker}: {e}")

--- Optimization for Forex and Indices (Pure Heikin Ashi) ---

Оптимизация Pure Smoothed HA для: EURUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 19
Итоговая доходность: 6.58%

Оптимизация Pure Smoothed HA для: GBPUSD=X


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 8
Итоговая доходность: 5.39%

Оптимизация Pure Smoothed HA для: AUDUSD=X


...

/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 3
Итоговая доходность: 9.49%

Оптимизация Pure Smoothed HA для: ^IXIC
Лучший период сглаживания: 19
Итоговая доходность: 36.41%

Оптимизация Pure Smoothed HA для: ^GSPC


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)
/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


Лучший период сглаживания: 19
Итоговая доходность: 21.78%

Оптимизация Pure Smoothed HA для: ^DJI
Лучший период сглаживания: 14
Итоговая доходность: 17.13%


/tmp/ipykernel_2823/4206796599.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period=period, interval='1d', progress=False)


In [19]:
def optimize_pure_ha_hourly(ticker):
    print(f"\n--- 1H Optimization (Pure HA) for: {ticker} ---")
    # Download 1h data (yfinance limit is approx 730 days)
    data = yf.download(ticker, period='730d', interval='1h', progress=False)
    if data.empty: return
    if isinstance(data.columns, pd.MultiIndex): data.columns = data.columns.get_level_values(0)

    smooth_ranges = range(2, 31) # Slightly wider range for hourly precision
    best_ret = 0
    best_smooth = None

    for sm in smooth_ranges:
        res = backtest_only_ha(data, sm, commission=0.001)
        if res > best_ret:
            best_ret = res
            best_smooth = sm

    print(f"Best 1H Smoothing Period: {best_smooth}")
    print(f"1H Strategy Return: {(best_ret-1)*100:.2f}%")

# Run optimization for all assets on 1H timeframe
for ticker in all_requested_assets:
    try:
        optimize_pure_ha_hourly(ticker)
    except Exception as e:
        print(f"Error optimizing {ticker}: {e}")


--- 1H Optimization (Pure HA) for: EURUSD=X ---


/tmp/ipykernel_2823/3785441673.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Best 1H Smoothing Period: 29
1H Strategy Return: -63.89%

--- 1H Optimization (Pure HA) for: GBPUSD=X ---


/tmp/ipykernel_2823/3785441673.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Best 1H Smoothing Period: 28
1H Strategy Return: -63.56%

--- 1H Optimization (Pure HA) for: AUDUSD=X ---


...

Best 1H Smoothing Period: 28
1H Strategy Return: 18.33%

--- 1H Optimization (Pure HA) for: ^GSPC ---


/tmp/ipykernel_2823/3785441673.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Best 1H Smoothing Period: 30
1H Strategy Return: 1.22%

--- 1H Optimization (Pure HA) for: ^DJI ---


/tmp/ipykernel_2823/3785441673.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, period='730d', interval='1h', progress=False)


Best 1H Smoothing Period: 26
1H Strategy Return: -2.21%


In [20]:
specific_params = (5, 10, 15)
print(f"Проверка пользовательских настроек: {specific_params}\n")

for asset in assets:
    data = yf.download(asset, period='2y', interval='1d', progress=False)
    if not data.empty:
        ret = backtest_macd(data, *specific_params)
        print(f"{asset}: Доходность = {(ret-1)*100:.2f}%")

Проверка пользовательских настроек: (5, 10, 15)

BTC-USD: Доходность = 16.34%


/tmp/ipykernel_2823/1475501505.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='2y', interval='1d', progress=False)
/tmp/ipykernel_2823/1475501505.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='2y', interval='1d', progress=False)


ETH-USD: Доходность = 20.76%
EURUSD=X: Доходность = -4.05%


/tmp/ipykernel_2823/1475501505.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='2y', interval='1d', progress=False)
/tmp/ipykernel_2823/1475501505.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(asset, period='2y', interval='1d', progress=False)


GBPUSD=X: Доходность = -5.29%
